<a href="https://colab.research.google.com/github/soumoreira/Inteligencias_Artificiais_Generativas_Aplicada_a_Programacao/blob/main/farmac%C3%AAutico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chatbot farmacêutico interativo.

projetado para simular um farmacêutico cordial e preciso que pode responder a perguntas sobre interações medicamentosas.

In [9]:
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, HTML

# Carrega a chave da API do Gemini do Secrets Manager
try:
    GOOGLE_API_KEY = userdata.get('Famacia')
    genai.configure(api_key=GOOGLE_API_KEY)
except userdata.SecretNotFoundError:
    print("ATENÇÃO: A chave 'Famacia' não foi encontrada nos segredos do Colab.")
    print("Por favor, verifique se você adicionou sua chave de API do Gemini com o nome 'Famacia' no Secrets Manager do Colab.")
    print("Vá em '🔑' no painel esquerdo, clique em 'Adicionar novo segredo' e use 'Famacia' como nome.")
    GOOGLE_API_KEY = None

In [10]:
# Define a persona do farmacêutico
pharmacist_persona = """Você é um farmacêutico cordial, atencioso e muito preciso.
Sua principal função é esclarecer dúvidas sobre interações medicamentosas.
Ao responder, você deve:
1. Sempre manter um tom acolhedor e profissional.
2. Ser extremamente preciso nas informações sobre interações.
3. Fazer perguntas pertinentes sobre outros medicamentos que a pessoa esteja tomando,
   além de comorbidades (outras condições médicas) relevantes para a situação.
   Isso é crucial para garantir a segurança e a eficácia da medicação.
4. Se não tiver certeza ou precisar de mais informações, peça clareza e nunca
   forneça conselhos incompletos ou potencialmente perigosos."""

# Inicializa o modelo Gemini e inicia uma sessão de chat com a persona
if GOOGLE_API_KEY:
    model = genai.GenerativeModel('gemini-pro-latest')
    chat = model.start_chat(history=[{'role':'user', 'parts': pharmacist_persona}, {'role':'model', 'parts': 'Entendido! Estou pronto para ajudar com suas dúvidas sobre interações medicamentosas. Por favor, comece me informando quais medicamentos você está tomando e sua dúvida principal.'}])
else:
    print("A API do Gemini não foi configurada. Por favor, adicione sua chave de API.")
    chat = None

In [ ]:
if chat:
    # Cria os widgets para a interface do chat
    chat_output = widgets.Output()
    text_input = widgets.Text(placeholder='Digite sua pergunta aqui...', layout=widgets.Layout(width='80%'))
    send_button = widgets.Button(description='Enviar', layout=widgets.Layout(width='15%'))

    def on_send_button_clicked(b):
        with chat_output:
            user_message = text_input.value
            if user_message.strip() == '':
                return

            # Limpa o input
            text_input.value = ''
            display(HTML(f'<p><b>Você:</b> {user_message}</p>'))

            try:
                response = chat.send_message(user_message)
                display(HTML(f'<p><b>Farmacêutico:</b> {response.text}</p>'))
            except Exception as e:
                display(HTML(f'<p style="color:red;"><b>Erro:</b> {e}</p>'))

    send_button.on_click(on_send_button_clicked)

    # Layout da interface
    chat_interface = widgets.VBox([
        chat_output,
        widgets.HBox([text_input, send_button])
    ])

    display(chat_interface)
else:
    display(HTML('<p style="color:red;">Não foi possível iniciar a interface do chat porque a API do Gemini não está configurada.</p>'))
